In [ ]:
# 手动模拟 _get_branch_path_input_schema 的调用过程
from langgraph.graph._branch import _get_branch_path_input_schema
from langchain_core.runnables import RunnableLambda
from typing import Any
from pydantic import BaseModel

# 测试不同的分支函数类型
def test_branch_function(state: dict[str, Any]) -> str:
    return "next_node"


class TestState(BaseModel):
    data: str


def typed_branch_function(state: TestState) -> str:
    return "typed_next"


# 测试调用 _get_branch_path_input_schema
print("测试普通函数:")
schema1 = _get_branch_path_input_schema(test_branch_function)
print(f"推断的输入模式: {schema1}")

print("\n测试有类型注解的函数:")
schema2 = _get_branch_path_input_schema(typed_branch_function)
print(f"推断的输入模式: {schema2}")

print("\n测试 RunnableLambda:")
runnable_branch = RunnableLambda(typed_branch_function)
schema3 = _get_branch_path_input_schema(runnable_branch)
print(f"推断的输入模式: {schema3}")

In [28]:
from langgraph.channels import LastValue, EphemeralValue
from langgraph.pregel import Pregel, NodeBuilder

# 节点1：从通道"a"读取，处理后写入通道"b"
node1 = (
    NodeBuilder().subscribe_only("a").do(lambda x: x + x).write_to("b")  # 将输入值翻倍
)

# 创建 Pregel 应用
app = Pregel(
    nodes={"node1": node1},
    channels={
        "a": EphemeralValue(str),  # 临时值通道
        "b": LastValue(str),  # 存储最后值的通道
    },
    input_channels=["a"],
    output_channels=["b"],
)

app.invoke({"a": "foo"})

# 运行机制：
# 1. 输入: {"a": "foo"}
# 2. Plan: 选择订阅"a"通道的节点（node1）
# 3. Execution: node1 执行，读取"foo"，计算"foofoo"
# 4. Update: 将"foofoo"写入通道"b"
# 5. 输出: {'b': 'foofoo'}

{'b': 'foofoo'}

In [29]:
from langgraph.channels import Topic, EphemeralValue
from langgraph.pregel import Pregel, NodeBuilder

# 节点1：处理输入并写入多个通道
node1 = (
    NodeBuilder()
    .subscribe_only("a")
    .do(lambda x: x + x)
    .write_to("b", "c")  # 同时写入通道b和c
)

# 节点2：从通道b读取，处理后写入通道c
node2 = NodeBuilder().subscribe_to("b").do(lambda x: x["b"] + x["b"]).write_to("c")

app = Pregel(
    nodes={"node1": node1, "node2": node2},
    channels={
        "a": EphemeralValue(str),
        "b": EphemeralValue(str),
        "c": Topic(str, accumulate=True),  # 累积模式的主题通道
    },
    input_channels=["a"],
    output_channels=["c"],
)

app.invoke({"a": "foo"})

# 运行机制：
# 1. 输入: {"a": "foo"}
# 2. Plan: 选择订阅"a"的节点（node1）
# 3. Execution: node1 执行，产生"foofoo"
# 4. Update: 将"foofoo"写入通道b和c
# 5. Plan: 选择订阅"b"的节点（node2）
# 6. Execution: node2 执行，读取"foofoo"，产生"foofoofoofoo"
# 7. Update: 将"foofoofoofoo"写入通道c（累积）
# 8. 输出: {'c': ['foofoo', 'foofoofoofoo']}

{'c': ['foofoo', 'foofoofoofoo']}